# Check the Filing Summaries

| Item | Description |
|---|---|
| Goal | Put the shard outputs of one T3.7.2 run (one input set, one summary family, one prompt version) together into the filing-summary table, and check them: one summary per filing of the set, none empty, all within the embedding limit, with the run records and the quirks seen in the smoke run counted on the whole set. |
| Input | `<scratch>/T3.7 - Firm Summaries/Interim/filing_summaries/<set>/<family>/<version>/shard_<k>.jsonl` and `shard_<k>.json` from T3.7.2; `output/T3.4.1/filings.csv`. |
| Output | `<scratch>/T3.7 - Firm Summaries/Processed/filing_summaries.parquet`: one row per filing, family, and prompt version; rows of other runs already in the file are kept, rows of this run are replaced. `output/T3.7.3/filing_summary_check_<set>_<family>_<version>.csv` (one row per shard) and `example_summaries_<set>_<family>_<version>.csv`. |
| Unit | A filing's summary under one family. |
| Checks | Every filing of the set has exactly one summary; no summary is empty; every summary is within 512 bge tokens; every shard and summary carries the run's prompt version. Counted, not enforced: the number of numbered lines per summary, whether a summary ends with an unnumbered closing sentence (the two quirks of the smoke run), and whether a list ends in a bare item number, which means the reduce call hit its output cap and the list was cut; and whether a summary refers to its own inputs ("the provided summaries do not describe", "this segment does not"), the leak seen in the first 1.3 shard. |

See `Topic Relevance Rebuild Plan.md` in the TEMP folder for the surrounding steps.


## Step 1. Import packages


In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


## Step 2. Define file locations and the run to check

Change `SET`, `FAMILY`, and `VERSION` to check another run; the table on scratch accumulates runs.


In [2]:
PROJECT_FOLDER = Path("/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis")
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP")
NOTEBOOK_SUBFOLDER = Path("T3 - Topic Relevance and Corporate Disclosure/TEMP")
NOTEBOOK_NAME = "T3.7.3"  # the identifier names this notebook's output folder
STAGE_FOLDER = SCRATCH_FOLDER / "T3.7 - Firm Summaries"
SUMMARIES_FOLDER = STAGE_FOLDER / "Interim" / "filing_summaries"
TABLE_FILE = STAGE_FOLDER / "Processed" / "filing_summaries.parquet"
FILINGS_FILE = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output/T3.4.1/filings.csv"
OUTPUT_FOLDER = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output" / NOTEBOOK_NAME

SET = "three_years"
FAMILY = "summary_lines_of_business"
VERSION = "item1-mapreduce-1.3"  # the prompt version the run was made with
MAX_BGE_TOKENS = 512  # the encoder's input limit every summary must respect
N_EXAMPLES = 6  # summaries shown and saved as examples
RANDOM_SEED = 0
TABLE_FILE.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Run: set {SET}, family {FAMILY}, prompt version {VERSION}")
print(f"Table: {TABLE_FILE}")
print(f"Outputs: {OUTPUT_FOLDER}")


Run: set three_years, family summary_lines_of_business, prompt version item1-mapreduce-1.3
Table: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP/T3.7 - Firm Summaries/Processed/filing_summaries.parquet
Outputs: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.7.3


## Step 3. Load the shard outputs and run records

Every shard file of the run is read; the run records give the GPU, the calls, and the time per shard.


In [3]:
run_folder = SUMMARIES_FOLDER / SET / FAMILY / VERSION
summary_files = sorted(path for path in run_folder.glob("shard_*.jsonl") if re.fullmatch(r"shard_\d+\.jsonl", path.name))  # not the .calls.jsonl logs
assert summary_files, f"no shard outputs under {run_folder}"
rows = []
for summary_file in summary_files:
    for line in summary_file.read_text(encoding="utf-8").splitlines():
        record = json.loads(line)
        record["shard_id"] = int(summary_file.stem.split("_")[1])
        rows.append(record)
summaries = pd.DataFrame(rows)
records = pd.DataFrame([json.loads(path.read_text(encoding="utf-8")) for path in sorted(run_folder.glob("shard_*.json")) if re.fullmatch(r"shard_\d+\.json", path.name)])
print(f"Shard files: {len(summary_files)}; summaries read: {len(summaries):,}; run records: {len(records)}")
display(records[["shard_id", "filings_done_this_run", "filings_skipped_as_done", "segments", "model_calls", "gpu", "seconds"]].assign(minutes=lambda d: (d["seconds"] / 60).round(1)).drop(columns="seconds"))


Shard files: 8; summaries read: 2,072; run records: 8


,shard_id,filings_done_this_run,filings_skipped_as_done,segments,model_calls,gpu,minutes
0,0,259,0,716,975,NVIDIA A40,31.3
1,1,259,0,788,1049,NVIDIA A40,33.8
2,2,259,0,692,953,NVIDIA A40,30.8
3,3,259,0,768,1027,NVIDIA A40,32.7
4,4,259,0,947,1208,NVIDIA A40,40.4
5,5,259,0,840,1099,NVIDIA A40,36.2
6,6,259,0,845,1105,NVIDIA A40,35.5
7,7,229,30,790,1023,NVIDIA A40,34.0


## Step 4. Check coverage against the sample

One summary per filing of the set, no filing missing and none repeated. A shard that was run in two sessions can hold a filing twice; the later record wins and the case is counted.


In [4]:
filings = pd.read_csv(FILINGS_FILE, dtype={"gvkey": str, "cik": str})
expected = filings.loc[filings["in_three_years"].eq(1) if SET == "three_years" else filings["in_three_years"].eq(0), "accessionNo"]
duplicates = int(summaries.duplicated("accessionNo").sum())
summaries = summaries.drop_duplicates("accessionNo", keep="last")
missing = sorted(set(expected) - set(summaries["accessionNo"]))
extra = sorted(set(summaries["accessionNo"]) - set(expected))
assert not extra, f"{len(extra)} summaries for filings outside the set"
print(f"Expected filings: {len(expected):,}; summaries: {len(summaries):,}; duplicate records dropped: {duplicates}; missing filings: {len(missing)}")
assert not missing, f"missing filings, for example {missing[:5]}"
assert summaries["prompt_version"].eq(VERSION).all()
assert records["prompt_version"].eq(VERSION).all()
assert summaries["family"].eq(FAMILY).all()


Expected filings: 2,072; summaries: 2,072; duplicate records dropped: 0; missing filings: 0


## Step 5. Check the summaries themselves

None empty and all within the embedding limit; the collapse levels and reduce re-asks show how often the budget rules bit.


In [5]:
assert summaries["summary"].str.strip().ne("").all()
assert summaries["within_bge_limit"].all()
print(f"Words per summary: min {summaries['n_words'].min()}, median {int(summaries['n_words'].median())}, max {summaries['n_words'].max()}")
print(f"bge tokens per summary: min {summaries['n_bge_tokens'].min()}, median {int(summaries['n_bge_tokens'].median())}, max {summaries['n_bge_tokens'].max()} (limit {MAX_BGE_TOKENS})")
print(f"Segments per filing: median {int(summaries['n_segments'].median())}, max {summaries['n_segments'].max()}; filings that needed a collapse round: {int(summaries['collapse_levels'].gt(0).sum()):,}; filings re-asked shorter: {int(summaries['reduce_rounds'].gt(1).sum()):,}")


Words per summary: min 91, median 193, max 396
bge tokens per summary: min 118, median 264, max 507 (limit 512)
Segments per filing: median 3, max 34; filings that needed a collapse round: 0; filings re-asked shorter: 9


## Step 6. Count the quirks seen in the smoke runs

The number of numbered lines per summary, whether the text continues after the last numbered line with an unnumbered closing sentence, and whether the list ends in a bare item number, the sign of a reduce call cut at its output cap (the list versions). For the paragraph versions, whether the summary refers to its own inputs instead of the company: "the provided summaries do not describe", "this segment does not", "no information is contained within this text". All are counted for the record, not enforced.


In [6]:
def count_numbered_lines(text):
    return len(re.findall(r"(?m)^\s*\d+[.)]\s", text))


def has_closing_sentence(text):
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    return bool(lines) and not re.match(r"^\d+[.)]\s", lines[-1])


def ends_truncated(text):
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    return bool(lines) and re.fullmatch(r"\d+[.)]", lines[-1]) is not None


REFERS_TO_INPUT = re.compile(
    r"\b(the|these|provided) (provided |segment |yearly )?summar(y|ies)\b"  # "the provided summaries do not describe"
    r"|\b(this|the) (text|segment)s? (does|do) not\b"  # "this segment does not describe"
    r"|contained within this (text|segment)|not (described|mentioned|provided|contained) in (the|this|these) (text|segment|summar)",
    re.I,
)


def refers_to_input(text):
    return REFERS_TO_INPUT.search(text) is not None


summaries["n_numbered_lines"] = summaries["summary"].map(count_numbered_lines)
summaries["has_closing_sentence"] = summaries["summary"].map(has_closing_sentence)
summaries["ends_truncated"] = summaries["summary"].map(ends_truncated)
summaries["refers_to_input"] = summaries["summary"].map(refers_to_input)
print(f"Lists cut at the output cap (ending in a bare item number): {int(summaries['ends_truncated'].sum()):,} of {len(summaries):,}")
print(f"Numbered lines per summary: min {summaries['n_numbered_lines'].min()}, median {int(summaries['n_numbered_lines'].median())}, max {summaries['n_numbered_lines'].max()}; summaries with no numbered line: {int(summaries['n_numbered_lines'].eq(0).sum()):,}")
print(f"Summaries ending with an unnumbered closing sentence: {int(summaries['has_closing_sentence'].sum()):,} of {len(summaries):,} ({summaries['has_closing_sentence'].mean():.1%})")
print(f"Summaries referring to their own inputs: {int(summaries['refers_to_input'].sum()):,} of {len(summaries):,} ({summaries['refers_to_input'].mean():.1%})")
for text in summaries.loc[summaries["refers_to_input"], "summary"].head(3):
    print("  e.g.:", REFERS_TO_INPUT.search(text).group(0), "...", text[-160:].replace("\n", " "))
display(summaries["n_numbered_lines"].value_counts().sort_index().rename("summaries").to_frame().T)


Lists cut at the output cap (ending in a bare item number): 0 of 2,072
Numbered lines per summary: min 0, median 0, max 0; summaries with no numbered line: 2,072
Summaries ending with an unnumbered closing sentence: 2,072 of 2,072 (100.0%)
Summaries referring to their own inputs: 31 of 2,072 (1.5%)
  e.g.: the provided summaries ... ial and institutional entities, and internal operational activities such as labor relations or regulatory compliance are not detailed in the provided summaries.
  e.g.: the provided summaries ... ncial service offerings across its designated geographic markets without any other business segments or customer categories mentioned in the provided summaries.
  e.g.: the summaries ... st other countries. The company's business encompasses these core energy and chemical activities without further segmentation details provided in the summaries.


n_numbered_lines,0
summaries,2072


## Step 7. Show examples

Random summaries, with the shortest and the longest, so the reader sees the range.


In [7]:
rng = np.random.default_rng(RANDOM_SEED)
chosen = pd.concat([
    summaries.iloc[rng.choice(len(summaries), size=N_EXAMPLES - 2, replace=False)],
    summaries.nsmallest(1, "n_words"),
    summaries.nlargest(1, "n_words"),
])
examples = chosen[["accessionNo", "conm", "fyear", "n_segments", "n_words", "n_numbered_lines", "has_closing_sentence", "ends_truncated", "refers_to_input", "summary"]]
examples.to_csv(OUTPUT_FOLDER / f"example_summaries_{SET}_{FAMILY}_{VERSION}.csv", index=False)
pd.set_option("display.max_colwidth", 600)
for example in examples.itertuples(index=False):
    print(f"=== {example.conm} {example.fyear} | {example.n_segments} segments | {example.n_words} words | {example.n_numbered_lines} lines ===\n{example.summary}\n")


=== S&P GLOBAL INC 2020 | 1 segments | 160 words | 0 lines ===
S&P Global Inc. operates as a leading provider of transparent ratings, benchmarks, analytics, and data to the capital and commodity markets worldwide through four distinct reportable segments. The Ratings segment offers credit ratings and research services to investors, corporations, governments, and financial institutions. Market Intelligence provides data, analytics, and third-party research specifically to investment managers, banks, and corporations. Platts supplies benchmark prices and market data to producers, traders, and intermediaries within the energy, metals, and agriculture sectors. The Indices segment maintains various indices to generate asset-linked fees for investors and fund managers. The company serves its global customer base through both third-party and proprietary distribution channels, delivering a diverse array of products including credit ratings, subscription-based data feeds, real-time news, sales 

## Step 8. Save the table and the check record

Rows of other runs already in the table are kept; rows of this set, family, and version are replaced by this run's.


In [8]:
table_columns = ["accessionNo", "gvkey", "conm", "fyear", "set", "family", "prompt_version", "summary",
                 "n_words", "n_bge_tokens", "n_segments", "collapse_levels", "reduce_rounds", "n_numbered_lines", "has_closing_sentence", "ends_truncated", "refers_to_input", "run_started"]
new_rows = summaries.assign(set=SET)[table_columns]
if TABLE_FILE.exists():
    existing = pd.read_parquet(TABLE_FILE)
    existing = existing.loc[~(existing["set"].eq(SET) & existing["family"].eq(FAMILY) & existing["prompt_version"].eq(VERSION))]
    table = pd.concat([existing, new_rows], ignore_index=True)
else:
    table = new_rows
table["refers_to_input"] = table["summary"].map(refers_to_input)  # filled for the rows of earlier runs too, so the versions compare
assert not table.duplicated(["accessionNo", "family", "prompt_version"]).any()
table.to_parquet(TABLE_FILE, index=False)
check = records[["shard_id", "filings_done_this_run", "filings_skipped_as_done", "segments", "model_calls", "gpu", "seconds", "prompt_version"]].copy()
check["set"], check["family"] = SET, FAMILY
check["n_summaries_in_run"] = len(summaries)
check["closing_sentence_share"] = float(summaries["has_closing_sentence"].mean())
check["truncated_share"] = float(summaries["ends_truncated"].mean())
check["refers_to_input_share"] = float(summaries["refers_to_input"].mean())
check.to_csv(OUTPUT_FOLDER / f"filing_summary_check_{SET}_{FAMILY}_{VERSION}.csv", index=False)
print(f"Saved: {TABLE_FILE} ({len(table):,} rows; this run {len(new_rows):,})")
print(f"Saved: {OUTPUT_FOLDER / f'filing_summary_check_{SET}_{FAMILY}_{VERSION}.csv'}")
print(table.groupby(["set", "family", "prompt_version"]).size().rename("rows").to_string())


Saved: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP/T3.7 - Firm Summaries/Processed/filing_summaries.parquet (6,216 rows; this run 2,072)
Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.7.3/filing_summary_check_three_years_summary_lines_of_business_item1-mapreduce-1.3.csv
set          family                     prompt_version     
three_years  summary_lines_of_business  item1-mapreduce-1.1    2072
                                        item1-mapreduce-1.2    2072
                                        item1-mapreduce-1.3    2072
